# Company pipeline - end-to-end test suite

Runs the **production pipeline code unchanged** against synthetic CSV files through the TEST pipeline **Trust360_Company_Medallion_TEST** (source `/Volumes/team_2/hr_company_test/test_data/OUT`, target schema `team_2.hr_company_test`).

Execution order:
1. Run cells 2-3 (config + stage 1 files) -> start a TEST pipeline update
2. Run cell 4 (stage 2: new snapshot with an extra column) -> start an update
3. Run cell 5 (stage 3: late-arriving snapshot + re-delivered file) -> start an update
4. Run cells 6-9 (assertions). Cell 9 snapshots Gold, then start one more update with **no new files** and run cell 10 (idempotency).

Every assertion prints PASS/FAIL; a failing assertion raises at the end of the cell.

In [0]:
TEST_ROOT = "/Volumes/team_2/hr_company_test/test_data/OUT"
T = "team_2.hr_company_test"
HEADER = "Company_ID,Company_Name,Location_ID,Is_InternalCompany"

def write_csv(rel_path: str, lines: list):
    """Write one synthetic CSV (UTF-8, LF) below the TEST source root (overwrites)."""
    dbutils.fs.put(f"{TEST_ROOT}/{rel_path}", "\n".join(lines) + "\n", overwrite=True)

_failures = []
def check(name: str, actual, expected):
    ok = actual == expected
    print(("PASS " if ok else "FAIL ") + name + ("" if ok else f"\n     expected={expected}\n     actual  ={actual}"))
    if not ok:
        _failures.append(name)

def finish():
    global _failures
    failed, _failures = _failures, []
    assert not failed, f"{len(failed)} assertion(s) failed: {failed}"
    print("ALL ASSERTIONS PASSED")

In [0]:
# 2026-01-01: one record per Silver test case (+ two malformed rows)
write_csv("2026/01/01/Company.csv", [
    HEADER,
    "T001,Alpha Ltd,LOC001,Y",               # S01 valid
    "COMPANY01,Beta & Co.,LOC002,N",         # S01 valid (alphanumeric ID)
    "123ABC,Gamma GmbH,LOC003,Y",            # S01 valid (digits first allowed)
    ",No Id Corp,LOC004,Y",                  # S02 COMPANY_ID_NULL
    "   ,Space Id Corp,LOC005,N",            # S03 COMPANY_ID_BLANK
    "COMP-001,Dash Corp,LOC006,Y",           # S04 COMPANY_ID_INVALID_FORMAT
    "COMP 001,Space Corp,LOC007,Y",          # S04 COMPANY_ID_INVALID_FORMAT
    "DUP1,Dup A,LOC008,Y",                   # S05 duplicate within the same file
    "DUP1,Dup B,LOC009,N",                   # S05
    "T010,,LOC010,Y",                        # S07 COMPANY_NAME_NULL
    "T011,   ,LOC011,Y",                     # S08 COMPANY_NAME_BLANK
    "T012,Name12,,Y",                        # S09 LOCATION_ID_NULL
    "T013,Name13,   ,N",                     # S10 LOCATION_ID_BLANK
    "T014,Name14,LOC-014,N",                 # S11 LOCATION_ID_INVALID_FORMAT
    "T015,Name15,LOC 015,N",                 # S11
    "T016,Name16,LOC016,",                   # S12 INTERNAL_COMPANY_NULL
    "T017,Name17,LOC017,   ",                # S13 INTERNAL_COMPANY_BLANK
    "T018,Name18,LOC018,YES",                # S14 INTERNAL_COMPANY_INVALID_VALUE
    "T019,Name19,LOC019,TRUE",               # S14
    "T020,Name20,LOC020,1",                  # S14
    "T021,Name21,LOC021,y",                  # S15 lowercase (case-sensitive default)
    "T022,,,X",                              # S16 multiple rules
    "  T023  ,  Société  Générale  ,LOC023,N",  # S17 whitespace + accents -> valid, trimmed
    "T024,Name24,LOC9999,Y",                 # S18 unknown Location (warn only)
    "T025,Name25,LOC025,Y,EXTRA",            # B09 malformed: too many fields
    "T026,Name26",                           # B09 malformed: too few fields
    "T030,Name30,LOC030,Y",                  # G05 flag will change
    "XD1,Cross A,LOC031,Y",                  # S06 duplicate across files (part 1)
])
write_csv("2026/01/01/Company_part2.csv", [
    HEADER,
    "XD1,Cross B,LOC032,N",                  # S06 duplicate across files (part 2)
    "T031,Name31,LOC031,Y",
])
# Not a Company file -> must be ignored by the file glob (B02)
write_csv("2026/01/01/Location.csv", ["Location_ID,Location_Title", "LOC001,Paris"])

# 2026-01-02 / 2026-01-03: SCD2 cases
write_csv("2026/01/02/Company.csv", [
    HEADER,
    "T001,Alpha Ltd,LOC001,Y",               # G02 unchanged
    "COMPANY01,Beta & Company,LOC002,N",     # G03 Company_Name change
    "123ABC,Gamma GmbH,LOC003,Y",
    "T023,Société Générale,LOC023,N",      # unchanged after standardisation
    "T030,Name30,LOC030,N",                  # G05 Is_InternalCompany change
    "T040,New Co,LOC040,Y",                  # G01 new Company in a later snapshot
    "DUP1,Dup A,LOC008,Y",                   # first valid appearance -> NEW
])                                           # T024 absent -> G07 must stay current
write_csv("2026/01/03/Company.csv", [
    HEADER,
    "T001,Alpha Ltd,LOC001,Y",
    "COMPANY01,Beta & Company,LOC002,N",
    "123ABC,Gamma GmbH,LOC033,Y",            # G04 Location_ID change
    "T023,Société Générale,LOC023,N",
    "T030,Name30,LOC030,N",
    "T040,New Co,LOC040,Y",
    "DUP1,Dup A,LOC008,Y",
])
# datephoto fallbacks (B05): date only in the file name / no date at all
write_csv("adhoc/Company_20260106.csv", [HEADER, "T091,FileName Date Co,LOC091,Y"])
write_csv("adhoc/Company_nodate.csv", [HEADER, "T092,Delivery Date Co,LOC092,N"])
print("stage 1 written -> start a TEST pipeline update")

In [0]:
# 2026-01-05: new file arrival (B03) with an additional column Company_Type (B10)
write_csv("2026/01/05/Company.csv", [
    HEADER + ",Company_Type",
    "T001,Alpha Ltd v2,LOC001,Y,HOLDING",    # G06 change (later back-dated by stage 3)
    "COMPANY01,Beta & Company,LOC002,N,SUB",
    "123ABC,Gamma GmbH,LOC033,Y,SUB",
    "T023,Société Générale,LOC023,N,SUB",
    "T030,Name30,LOC030,N,SUB",
    "T040,New Co,LOC040,Y,SUB",
    "DUP1,Dup A,LOC008,Y,SUB",
])
print("stage 2 written -> start a TEST pipeline update")

In [0]:
import time
# G09 late-arriving: 2026-01-04 arrives AFTER 2026-01-05 was processed
write_csv("2026/01/04/Company.csv", [
    HEADER,
    "T001,Alpha Ltd v2,LOC001,Y",            # change actually happened on 01-04
    "COMPANY01,Beta & Co.,LOC002,N",         # temporary revert (A->B->A->B)
    "123ABC,Gamma GmbH,LOC033,Y",
    "T023,Société Générale,LOC023,N",
    "T030,Name30,LOC030,N",
    "T040,New Co,LOC040,Y",
    "DUP1,Dup A,LOC008,Y",
])
# G08 duplicate snapshot processing: identical 2026-01-02 file re-delivered (new mtime)
time.sleep(2)
write_csv("2026/01/02/Company.csv", [
    HEADER,
    "T001,Alpha Ltd,LOC001,Y",
    "COMPANY01,Beta & Company,LOC002,N",
    "123ABC,Gamma GmbH,LOC003,Y",
    "T023,Société Générale,LOC023,N",
    "T030,Name30,LOC030,N",
    "T040,New Co,LOC040,Y",
    "DUP1,Dup A,LOC008,Y",
])
print("stage 3 written -> start a TEST pipeline update")

In [0]:
from pyspark.sql import functions as F

b = spark.table(f"{T}.bronze_company")
agg = (b.groupBy("source_file").agg(
        F.count("*").alias("n"), F.countDistinct("datephoto").alias("dp"), F.min("datephoto").alias("datephoto"),
        F.min("datephoto_source").alias("src"),
        F.countDistinct("source_file_modification_time").alias("versions"))
      .withColumn("rel", F.regexp_extract("source_file", "/OUT/(.*)$", 1)))
files = {r.rel: r for r in agg.collect()}

check("B01/B02 only Company*.csv files ingested (Location.csv ignored)",
      sorted(files), sorted(["2026/01/01/Company.csv", "2026/01/01/Company_part2.csv", "2026/01/02/Company.csv",
                             "2026/01/03/Company.csv", "2026/01/04/Company.csv", "2026/01/05/Company.csv",
                             "adhoc/Company_20260106.csv", "adhoc/Company_nodate.csv"]))
check("B08 no records dropped: 01-01 file rows (28 data lines incl. malformed)", files["2026/01/01/Company.csv"].n, 28)
check("B07 re-delivered 01-02 file kept as 2 versions in Bronze (raw history)", files["2026/01/02/Company.csv"].versions, 2)
check("B05 one datephoto per file", max(r.dp for r in files.values()), 1)
check("B05 datephoto from folder", str(files["2026/01/03/Company.csv"].datephoto), "2026-01-03")
check("B05 datephoto from file name", (str(files["adhoc/Company_20260106.csv"].datephoto), files["adhoc/Company_20260106.csv"].src), ("2026-01-06", "FILE_NAME"))
check("B05 datephoto fallback = delivery date", files["adhoc/Company_nodate.csv"].src, "FILE_MODIFICATION_TIME")
check("B04 original columns preserved", [c for c in b.columns if c in ("Company_ID","Company_Name","Location_ID","Is_InternalCompany")],
      ["Company_ID", "Company_Name", "Location_ID", "Is_InternalCompany"])
check("B04 raw whitespace kept in Bronze", b.filter("Company_Name = '  Société  Générale  '").count(), 1)
check("B06 metadata columns present", all(c in b.columns for c in ["datephoto", "source_file", "ingestion_timestamp"]), True)
check("B06 ingestion_timestamp never NULL", b.filter("ingestion_timestamp IS NULL").count(), 0)
check("B09 malformed rows kept and flagged", b.filter("Company_ID IN ('T025','T026') AND (_corrupt_record IS NOT NULL OR _rescued_data IS NOT NULL)").count(), 2)
check("B10 schema evolution: Company_Type added", "Company_Type" in b.columns, True)
check("B10 Company_Type NULL for older files", b.filter("datephoto = '2026-01-01' AND Company_Type IS NOT NULL").count(), 0)
finish()

In [0]:
e = spark.table(f"{T}.silver_company_errors").filter("datephoto = '2026-01-01'")
s = spark.table(f"{T}.silver_company")
actual = {(r.Company_ID, r.Company_Name): r.error_reason for r in e.collect()}
expected = {
    (None, "No Id Corp"): "COMPANY_ID_NULL",
    ("   ", "Space Id Corp"): "COMPANY_ID_BLANK",
    ("COMP-001", "Dash Corp"): "COMPANY_ID_INVALID_FORMAT",
    ("COMP 001", "Space Corp"): "COMPANY_ID_INVALID_FORMAT",
    ("DUP1", "Dup A"): "COMPANY_ID_DUPLICATE",
    ("DUP1", "Dup B"): "COMPANY_ID_DUPLICATE",
    ("XD1", "Cross A"): "COMPANY_ID_DUPLICATE",
    ("XD1", "Cross B"): "COMPANY_ID_DUPLICATE",
    ("T010", None): "COMPANY_NAME_NULL",
    ("T011", "   "): "COMPANY_NAME_BLANK",
    ("T012", "Name12"): "LOCATION_ID_NULL",
    ("T013", "Name13"): "LOCATION_ID_BLANK",
    ("T014", "Name14"): "LOCATION_ID_INVALID_FORMAT",
    ("T015", "Name15"): "LOCATION_ID_INVALID_FORMAT",
    ("T016", "Name16"): "INTERNAL_COMPANY_NULL",
    ("T017", "Name17"): "INTERNAL_COMPANY_BLANK",
    ("T018", "Name18"): "INTERNAL_COMPANY_INVALID_VALUE",
    ("T019", "Name19"): "INTERNAL_COMPANY_INVALID_VALUE",
    ("T020", "Name20"): "INTERNAL_COMPANY_INVALID_VALUE",
    ("T021", "Name21"): "INTERNAL_COMPANY_INVALID_VALUE",
    ("T022", None): "COMPANY_NAME_NULL, LOCATION_ID_NULL, INTERNAL_COMPANY_INVALID_VALUE",
    ("T025", "Name25"): "MALFORMED_RECORD",
    ("T026", "Name26"): "LOCATION_ID_NULL, INTERNAL_COMPANY_NULL, MALFORMED_RECORD",
}
for k in sorted(set(expected) | set(actual), key=str):
    check(f"S error_reason {k}", actual.get(k), expected.get(k))

valid_0101 = sorted(r.Company_ID for r in s.filter("datephoto = '2026-01-01'").collect())
check("S01/S17/S18 valid records 2026-01-01", valid_0101, sorted(["T001", "COMPANY01", "123ABC", "T023", "T024", "T030", "T031"]))
check("S17 whitespace standardised in Silver",
      s.filter("Company_ID = 'T023' AND datephoto = '2026-01-01'").select("Company_Name").first()[0], "Société Générale")
check("S18 unknown Location -> warning only (kept valid, NOT_FOUND)",
      s.filter("Company_ID = 'T024'").select("location_ref_status").first()[0], "NOT_FOUND")
check("S19 multiple reasons: rule_name lists all rules",
      e.filter("Company_ID = 'T022'").select("rule_name").first()[0], "R2_COMPANY_NAME, R3_LOCATION_ID, R4_IS_INTERNAL_COMPANY")
err_all = spark.table(f"{T}.silver_company_errors")
check("S20 error table has required columns",
      all(c in err_all.columns for c in ["Company_ID","Company_Name","Location_ID","Is_InternalCompany","datephoto","source_file",
                                          "ingestion_timestamp","error_reason","error_timestamp","run_date","rule_name"]), True)
check("S21 one error row per source record (no duplicates after re-delivery / reruns)",
      err_all.count(), err_all.select("source_file", "Company_ID", "Company_Name", "Location_ID", "Is_InternalCompany").distinct().count())
check("S21 Silver unique per Company_ID x datephoto (re-delivered 01-02 not double counted)",
      s.groupBy("Company_ID", "datephoto").count().filter("count > 1").count(), 0)
check("S21 01-02 valid rows after re-delivery", s.filter("datephoto = '2026-01-02'").count(), 7)
finish()

In [0]:
from pyspark.sql import Window, functions as F

g = spark.table(f"{T}.gold_company")
def hist(cid):
    return [(r.Company_Name, r.Location_ID, r.Is_InternalCompany, str(r.effective_start_date), str(r.effective_end_date), r.is_current)
            for r in g.filter(F.col("Company_ID") == cid).orderBy("effective_start_date").collect()]
OPEN = "9999-12-31"
check("G01 new Company in later snapshot", hist("T040"), [("New Co", "LOC040", "Y", "2026-01-02", OPEN, True)])
check("G02 unchanged snapshots -> single version (T023 incl. standardised whitespace)",
      hist("T023"), [("Société Générale", "LOC023", "N", "2026-01-01", OPEN, True)])
check("G03 Company_Name change + G10 multiple snapshots (A->B->A->B)", hist("COMPANY01"), [
    ("Beta & Co.", "LOC002", "N", "2026-01-01", "2026-01-01", False),
    ("Beta & Company", "LOC002", "N", "2026-01-02", "2026-01-03", False),
    ("Beta & Co.", "LOC002", "N", "2026-01-04", "2026-01-04", False),
    ("Beta & Company", "LOC002", "N", "2026-01-05", OPEN, True)])
check("G04 Location_ID change", hist("123ABC"), [
    ("Gamma GmbH", "LOC003", "Y", "2026-01-01", "2026-01-02", False),
    ("Gamma GmbH", "LOC033", "Y", "2026-01-03", OPEN, True)])
check("G05 Is_InternalCompany change", hist("T030"), [
    ("Name30", "LOC030", "Y", "2026-01-01", "2026-01-01", False),
    ("Name30", "LOC030", "N", "2026-01-02", OPEN, True)])
check("G09 late-arriving 01-04 snapshot back-dates the change", hist("T001"), [
    ("Alpha Ltd", "LOC001", "Y", "2026-01-01", "2026-01-03", False),
    ("Alpha Ltd v2", "LOC001", "Y", "2026-01-04", OPEN, True)])
check("G07 Company missing from later snapshots stays current", hist("T024"), [("Name24", "LOC9999", "Y", "2026-01-01", OPEN, True)])
check("G-dup all-duplicate snapshot excluded, first valid snapshot = start", hist("DUP1"), [("Dup A", "LOC008", "Y", "2026-01-02", OPEN, True)])
check("G-dup cross-file duplicate never reaches Gold", hist("XD1"), [])
check("G08 re-delivered 01-02 created no extra versions", g.filter("effective_start_date = '2026-01-02'").count(), 4)
check("G11 exactly one current row per Company_ID", g.groupBy("Company_ID").agg(F.sum(F.col("is_current").cast("int")).alias("c")).filter("c <> 1").count(), 0)
check("G12 datephoto/technical columns excluded from Gold", g.columns,
      ["Company_ID", "Company_Name", "Location_ID", "Is_InternalCompany", "effective_start_date", "effective_end_date", "is_current"])
w = Window.partitionBy("Company_ID").orderBy("effective_start_date")
check("G13 contiguous, non-overlapping intervals (end = next start - 1)",
      g.withColumn("nxt", F.lead("effective_start_date").over(w)).filter("nxt IS NOT NULL AND effective_end_date <> date_sub(nxt, 1)").count(), 0)
finish()

In [0]:
d = spark.table(f"{T}.dq_company_summary").filter("datephoto = '2026-01-01'").first().asDict()
check("M bronze_records_ingested (2 files)", (d["bronze_records_ingested"], d["source_files"]), (30, 2))
check("M valid + rejected = ingested", d["silver_valid_records"] + d["rejected_records"], d["bronze_records_ingested"])
check("M per-rule counts", {k: d[k] for k in ["invalid_company_id_records", "invalid_company_name_records", "invalid_location_id_records",
                                              "invalid_is_internal_company_records", "duplicate_company_id_records", "malformed_records",
                                              "location_ref_not_found_records"]},
      {"invalid_company_id_records": 4, "invalid_company_name_records": 3, "invalid_location_id_records": 6,
       "invalid_is_internal_company_records": 8, "duplicate_company_id_records": 4, "malformed_records": 2,
       "location_ref_not_found_records": 1})
check("M new companies on 01-01", d["new_companies"], 7)
finish()

In [0]:
# Persist the current Gold/Silver state as DATA (not a view), then start a TEST pipeline
# update WITHOUT adding files and run the next cell.
spark.table(f"{T}.gold_company").write.mode("overwrite").saveAsTable(f"{T}._test_gold_before_rerun")
spark.table(f"{T}.silver_company_errors").drop("error_timestamp", "run_date") \
    .write.mode("overwrite").saveAsTable(f"{T}._test_errors_before_rerun")
print("snapshot saved -> start a TEST pipeline update with no new files")

In [0]:
def diff(a, b):
    return a.exceptAll(b).count() + b.exceptAll(a).count()
check("G14 Gold identical after rerun", diff(spark.table(f"{T}.gold_company"), spark.table(f"{T}._test_gold_before_rerun")), 0)
check("S21 error table identical after rerun (no duplicate error rows)",
      diff(spark.table(f"{T}.silver_company_errors").drop("error_timestamp", "run_date"), spark.table(f"{T}._test_errors_before_rerun")), 0)
finish()